# Analysis of the Boolean model of cell cycle by Sizek et al. - Advanced Analysis

In this jupyter notebook, we will perform advanced analysis of the cell cycle model including state transition probabilities, circuit analysis, and mutant studies.

In [ ]:
import maboss
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from tools import load_trajs, draw_graph_from_pandas, compute_circuits, compute_stg_counts

The model files are available in the Boolean_models/cell_cycle/ folder of the tutorial sample project.

**API Note:** We'll start by loading the same model used in Tutorial 3, then extend the analysis.

In [ ]:
# TODO: Load the cell cycle model (same as Tutorial 3)
# path = "./Boolean_models/cell_cycle/"
# bnd_file = os.path.join(path, "intracellular_model.bnd")
# cfg_file = os.path.join(path, "intracellular_model.cfg")

# TODO: Load the model and set up phenotype tracking
# sim = maboss.load(bnd_file, cfg_file)
# sim_phenotypes = sim.copy()
# sim_phenotypes.network.set_output(['G0G1_entry', 'S_entry', 'G2M_entry'])

## Advanced State Transition Analysis

We can compute the same matrix but instead of transition counts, we have the probabilities.

**API Notes:**
- `np.divide()` with broadcasting computes row-wise normalization
- `stg_counts.sum(axis=1)[:, np.newaxis]` sums each row and reshapes for broadcasting

In [ ]:
# TODO: First run the trajectory analysis from Tutorial 3
# sim_phenotypes_trajs = sim_phenotypes.copy()
# sim_phenotypes_trajs.update_parameters(display_traj=1, thread_count=1, max_time=480)
# res_phenotypes_trajs = sim_phenotypes_trajs.run()

# TODO: Process trajectories to get transition counts
# outputs_phenotype = ["G0G1_entry", "G2M_entry", "S_entry"]
# trajs, all_states = load_trajs(res_phenotypes_trajs._path, outputs_phenotype)
# stg_counts, state_ids, ids_state = compute_stg_counts(trajs, all_states)

# TODO: Convert counts to probabilities
# probas = pd.DataFrame(
#     data=np.divide(stg_counts,stg_counts.sum(axis=1)[:, np.newaxis]), 
#     index=state_ids.keys(), columns=state_ids.keys()
# )
# probas

The subsequent analysis we can do is look at the different observed lists of activation, starting from the \<nil\> state.

**API Note:** `draw_graph_from_pandas()` can visualize probability matrices as well as count matrices.

In [ ]:
# TODO: Draw probability transition graph
# draw_graph_from_pandas(probas)

**API Notes:**
- `compute_circuits()` finds all possible paths/circuits in the state transition graph
- `%time` measures execution time of the circuit computation
- The function returns a dictionary with probabilities as keys and paths as values

In [ ]:
# TODO: Compute all possible circuits starting from <nil> state
# %time paths_dict = compute_circuits(probas, ids_state, '<nil>', 0)

In [ ]:
# TODO: Display circuits ordered by probability
# for proba in sorted(paths_dict, reverse=True):
#     #if proba > 0.01:
#     print("%.2f : %s" % (proba, paths_dict[proba]))

We can see that the model is not perfect here : while most of the sequence are complete, a large proportion skips the G2M phase. We refer to these cycles as incomplete cell cycles.

## Analysis of mutants

We can then look at known mutant affecting the cell cycle, and simulate them to see how the model predicts them. We will also include Caspase 3 as an output node, to see how these mutants affects cell death, and we will simulate them for 480 hours to see long term effects.

First, we look at the wild type, to remind us of its behavior and allow us to compare with mutants below.

**API Notes:**
- `sim.copy()` creates a copy for mutant analysis
- `sim.network.set_output()` updates output nodes to include Casp3
- `sim.update_parameters()` sets longer simulation time for mutant analysis

In [ ]:
# TODO: Create mutant analysis simulation
# sim_mutants = sim_phenotypes.copy()

# TODO: Set output to include apoptosis marker
# sim_mutants.network.set_output(["G0G1_entry", "G2M_entry", "S_entry", "Casp3"])

# TODO: Set longer simulation time for mutant effects
# sim_mutants.update_parameters(max_time=480)

# TODO: Run wild-type baseline
# res_mutants = sim_mutants.run()
# res_mutants.plot_node_trajectory()

We first look at the Plk1-- mutant, by forcing the inactivation of Plk1 along the simulation.

**API Notes:**
- `sim.mutate(node, state)` applies a permanent mutation (OFF, ON)
- `"OFF"` forces the node to remain inactive throughout simulation
- `"ON"` forces the node to remain active throughout simulation

In [ ]:
# TODO: Create Plk1 knockout mutant
# mut_Plk1_OFF = sim_mutants.copy()

# TODO: Apply Plk1 mutation (forced OFF)

# TODO: Run Plk1 mutant simulation

# TODO: Plot Plk1 mutant results


We can see that we don't see any cells in G0G1 nor S phase : cells get stuck in G2M phase.
We can also look at the sequence of transitions for this mutant.

**API Note:** Trajectory analysis can be applied to mutants to see how mutations affect state transitions.

In [ ]:
# TODO: Create trajectory analysis for Plk1 mutant
# mut_Plk1_OFF_trajs = mut_Plk1_OFF.copy()

# TODO: Enable trajectory recording
# mut_Plk1_OFF_trajs.update_parameters(display_traj=1, thread_count=1, max_time=480)

# TODO: Run trajectory simulation
# res_mut_Plk1_OFF_trajs = mut_Plk1_OFF_trajs.run()

# TODO: Process mutant trajectories
# trajs, all_states = load_trajs(res_mut_Plk1_OFF_trajs._path, outputs_phenotype)
# stg_counts, state_ids, ids_state = compute_stg_counts(trajs, all_states)

# TODO: Create mutant transition matrix
# data_plk1 = pd.DataFrame(
#     data=stg_counts,
#     index=state_ids.keys(), columns=state_ids.keys()
# )
# data_plk1

In [ ]:
# TODO: Visualize Plk1 mutant state transitions
# draw_graph_from_pandas(data_plk1)

And indeed, we can see that most trajectories stop at this G2M_entry.

Then, we look at the FoxO3 mutant.

**API Note:** Different mutations can have drastically different effects on cell cycle progression.

In [ ]:
# TODO: Create FoxO3 knockout mutant
# mut_FoxO3_OFF = sim_mutants.copy()

# TODO: Apply FoxO3 mutation

# TODO: Run FoxO3 mutant simulation

# TODO: Plot FoxO3 mutant trajectories


We can see that the cell cycle stops after one or a few cycles.

**API Note:** Trajectory analysis helps understand why certain mutants stop cycling.

In [ ]:
# TODO: Create FoxO3 mutant trajectory analysis
# mut_FoxO3_OFF_trajs = mut_FoxO3_OFF.copy()

# TODO: Enable trajectory recording for FoxO3 mutant
# mut_FoxO3_OFF_trajs.update_parameters(display_traj=1, thread_count=1, max_time=480)

# TODO: Run trajectory simulation
# res_mut_FoxO3_OFF_trajs = mut_FoxO3_OFF_trajs.run()

# TODO: Process FoxO3 mutant trajectories
# trajs, all_states = load_trajs(res_mut_FoxO3_OFF_trajs._path, outputs_phenotype)
# stg_counts, state_ids, ids_state = compute_stg_counts(trajs, all_states)

# TODO: Create FoxO3 mutant transition matrix
# data_foxo3 = pd.DataFrame(
#     data=stg_counts,
#     index=state_ids.keys(), columns=state_ids.keys()
# )
# data_foxo3

In [ ]:
# TODO: Visualize FoxO3 mutant state transitions
# draw_graph_from_pandas(data_foxo3)

Here we can see that most trajectories goes back to the \<nil\>, and the cycle stops there.

Finally we look at the p110++ mutant.

**API Note:** `"ON"` mutations force constitutive activation of the target node.

In [ ]:
# TODO: Create p110 overexpression mutant
# mut_p110_ON = sim_mutants.copy()

# TODO: Apply p110 mutation (forced ON)

# TODO: Run p110 mutant simulation

# TODO: Plot p110 mutant trajectories


We don't see much effect on the cell cycle, but the apoptosis pathway is turned off.

**Summary:** This tutorial demonstrated advanced MaBoSS analysis including:
- State transition probability analysis
- Circuit computation for pathway analysis  
- Systematic mutant analysis with trajectory tracking
- Comparison of different mutation effects on cell cycle dynamics

The combination of probability analysis and trajectory tracking provides deep insights into how Boolean networks behave under different genetic perturbations.